## Init

**Purpose:** Loads the shared PySpark utilities used throughout this notebook — column expression helpers (`F`), the `StringType` definition, and common column functions (`trim`, `col`, `when`, `coalesce`, `lit`) used in the transformation section below.

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col, when, coalesce, lit


## Reading from the Bronze Table

**Purpose:** Loads the raw order-level sales data from the Bronze layer (`workspace.bronze.orders_data`) into a Spark DataFrame for cleaning.

**Schema:** `Date`, `Order_num`, `Status`, `Customer`, `Customer_type`, `Products`, `Items_sold`, `Coupons`, `Net_Sales`, `Attribution`, and `N_Revenue_formatted` — one row per order with its products, savings, coupon, and attribution details.

In [0]:
df = spark.table('workspace.bronze.orders_data')

## Data Transformations

**Purpose:** Cleans the bronze data and prepares it for the Silver layer.

**Steps:**
1. `Date` — convert timestamp to date-only (`yyyy-mm-dd`).
2. `Order_num` — rename to `Order_Number`.
3. `N_Revenue_formatted` — split into separate `Revenue_Amount` and `Savings_Amount` columns, then drop the original.
4. `Status`, `Customer`, `Customer_type` — trim whitespace and apply proper case.
5. `Products` — trim whitespace and decode HTML entities (e.g. `&amp;` → `&`).
6. `Coupons` — trim, replace nulls with `0`, and derive a `Coupon_Amount` from the known coupon codes (`ace410` = $10, `acesave5` = $5).
7. `Attribution` — trim whitespace.
8. Rename all columns to the snake_case standard (e.g. `order_date`, `order_number`, `customer_name`).

**Note:** `Items_sold` and `Net_Sales` already arrive in the correct format (integer and double), so they pass through unchanged.

In [0]:
# Transform columns one by one

# 1. Date - convert timestamp to date-only (yyyy-mm-dd)
df = df.withColumn('Date', F.to_date(col('Date')))

# 2. Order_num - rename to Order_Number
df = df.withColumnRenamed('Order_num', 'Order_Number')

# 3. N_Revenue_formatted - separate revenue and savings amounts
df = df.withColumn('N_Revenue_formatted', trim(col('N_Revenue_formatted')))

# Extract the main revenue amount (first dollar amount)
revenue_extracted = F.regexp_extract(col('N_Revenue_formatted'), r'\$([0-9]+\.[0-9]{2})', 1)
df = df.withColumn('Revenue_Amount', 
    when(revenue_extracted != '', revenue_extracted.cast('double')).otherwise(None))

# Extract savings amount (if exists, otherwise 0)
df = df.withColumn('Savings_Amount',
    when(col('N_Revenue_formatted').contains('You saved'),
         F.regexp_extract(col('N_Revenue_formatted'), r'You saved \$([0-9]+\.[0-9]{2})', 1).cast('double')
    ).when(col('N_Revenue_formatted').contains('$0.00'),
         F.regexp_extract(col('N_Revenue_formatted'), r'\s+\$([0-9]+\.[0-9]{2})$', 1).cast('double')
    ).otherwise(0))

# 4. Status - trim and lowercase for consistency
df = df.withColumn('Status', F.initcap(trim(col('Status'))))

# 5. Customer - trim whitespace and proper case
df = df.withColumn('Customer', F.initcap(trim(col('Customer'))))

# 6. Customer_type - trim and lowercase
df = df.withColumn('Customer_type', F.initcap(trim(col('Customer_type'))))

# 7. Products - trim and decode HTML entities
df = df.withColumn('Products', trim(col('Products')))
df = df.withColumn('Products', F.regexp_replace(col('Products'), '&amp;', '&'))

# 8. Items_sold - already in correct format (integer)
# No transformation needed

# 9. Coupons - trim whitespace, replace null with 0, and extract coupon amount
df = df.withColumn('Coupons', trim(col('Coupons')))
df = df.withColumn('Coupons', coalesce(col('Coupons'), lit('0')))

# Extract coupon amount from specific coupon codes only
df = df.withColumn('Coupon_Amount',
    when(col('Coupons') == 'ace410', 10)
    .when(col('Coupons') == 'acesave5', 5)
    .otherwise(0)
)

# 10. Net_Sales - already in correct format (double)
# No transformation needed

# 11. Attribution - trim whitespace
df = df.withColumn('Attribution', trim(col('Attribution')))

# Remove unwanted columns
df = df.drop('N_Revenue_formatted')

# Rename all columns to snake_case standard
df = df.select(
    col('Date').alias('order_date'),
    col('Order_Number').alias('order_number'),
    col('Status').alias('order_status'),
    col('Customer').alias('customer_name'),
    col('Customer_type').alias('customer_type'),
    col('Products').alias('products_name'),
    col('Items_sold').alias('items_sold'),
    col('Coupons').alias('coupon_code'),
    col('Net_Sales').alias('net_sales'),
    col('Attribution').alias('attribution'),
    col('Revenue_Amount').alias('revenue_amount'),
    col('Savings_Amount').alias('savings_amount'),
    col('Coupon_Amount').alias('coupon_amount')
)



In [0]:
df.display()

## Write Into Silver Table

**Purpose:** Promotes the cleaned orders data to the Silver layer by overwriting `workspace.silver.silver_orders_data` (1,577 rows). The SQL cell below then verifies the result against the new table.

In [0]:
# Write the transformed data to the silver table
df.write \
    .mode('overwrite') \
    .saveAsTable('workspace.silver.silver_orders_data')

print(f"✓ Successfully wrote {df.count()} rows to workspace.silver.silver_orders_data")

In [0]:
%sql

select * from workspace.silver.silver_orders_data

## Order Insights

**Purpose:** Turns the cleaned silver data into business answers across four angles:

1. **Overall KPIs** — period totals, order statuses, and averages
2. **Revenue by customer type** — New versus Returning customers
3. **Top 10 customers by revenue** — the most valuable accounts
4. **Attribution channels** — where orders and revenue come from

### 1. Overall KPIs

One-row summary of the full order history: total orders, date range, completed versus refunded orders, revenue collected, savings granted, coupon usage, and the average order value.

In [0]:
silver_df = spark.table('workspace.silver.silver_orders_data')

kpi_df = silver_df.agg(
    F.count('*').alias('total_orders'),
    F.min('order_date').alias('first_order'),
    F.max('order_date').alias('last_order'),
    F.sum(when(col('order_status') == 'Completed', 1).otherwise(0)).alias('completed_orders'),
    F.sum(when(col('order_status') == 'Refunded', 1).otherwise(0)).alias('refunded_orders'),
    F.round(F.sum('revenue_amount'), 2).alias('total_revenue'),
    F.round(F.sum('net_sales'), 2).alias('total_net_sales'),
    F.round(F.sum('savings_amount'), 2).alias('total_savings'),
    F.round(F.sum('coupon_amount'), 2).alias('total_coupon_discounts'),
    F.sum('items_sold').alias('total_items_sold'),
    F.round(F.avg('revenue_amount'), 2).alias('avg_order_value'),
)

display(kpi_df)

### 2. Revenue by Customer Type

New versus Returning customers: order counts, revenue, average order value, and share of total revenue. Returning customers dominate the order book, but this view quantifies by how much.

In [0]:
from pyspark.sql.window import Window

customer_type_df = (silver_df
    .groupBy('customer_type')
    .agg(
        F.count('*').alias('orders'),
        F.countDistinct('customer_name').alias('customers'),
        F.round(F.sum('revenue_amount'), 2).alias('revenue'),
        F.round(F.avg('revenue_amount'), 2).alias('avg_order_value'),
        F.round(F.sum('revenue_amount') / F.sum(F.sum('revenue_amount')).over(Window.orderBy(F.lit(1))) * 100, 2).alias('pct_of_revenue'),
    )
    .orderBy('revenue', ascending=False)
)

display(customer_type_df)

### 3. Top 10 Customers by Revenue

The ten highest-value accounts with their order counts, total revenue, and items bought — a view of who the business depends on most.

In [0]:
top_customers_df = (silver_df
    .groupBy('customer_name', 'customer_type')
    .agg(
        F.count('*').alias('orders'),
        F.round(F.sum('revenue_amount'), 2).alias('revenue'),
        F.sum('items_sold').alias('items_sold'),
    )
    .orderBy('revenue', ascending=False)
    .limit(10)
)

display(top_customers_df)

### 4. Attribution Channels

Orders and revenue grouped by acquisition channel. Direct and Organic: Google dominate, while Unknown covers orders with no recorded source.

In [0]:
attribution_df = (silver_df
    .groupBy('attribution')
    .agg(
        F.count('*').alias('orders'),
        F.round(F.sum('revenue_amount'), 2).alias('revenue'),
        F.round(F.avg('revenue_amount'), 2).alias('avg_order_value'),
    )
    .orderBy(F.col('revenue').desc())
)

display(attribution_df)